# conv2d_pitch_only.py -- training job (RunPod)

Runs 6-fold player-held-out cross-validation for `Conv2dPitchOnlyModel` (2D CNN backbone, pitch-only, no tab head).

**Before running:** this notebook's working directory must be the `fretify` project root on the pod (e.g. `/workspace/fretify`) -- that's where `foundational_heads.py`, `conv2d_pitch_only.py`, `preprocessing/`, `data/guitarset/annotation/`, `output/processed_cqt/`, and `output/per_frame_annotations/` need to already be present (rsync'd over before opening this). The cwd-check cell below will tell you if something's missing.

**Kernel:** just use whichever default kernel RunPod's Jupyter environment starts with -- if you picked a PyTorch/CUDA template, torch is already installed there system-wide, no separate kernel registration needed (that step was only relevant for running locally on a Mac alongside a separate TensorFlow venv, not on a dedicated pod).

In [ ]:
import os

# Uncomment and edit if this notebook isn't already running from the project root:
# os.chdir("/workspace/fretify")

print("cwd:", os.getcwd())
for required in ["foundational_heads.py", "conv2d_pitch_only.py", "preprocessing",
                  "data/guitarset/annotation", "output/processed_cqt", "output/per_frame_annotations"]:
    status = "OK" if os.path.exists(required) else "MISSING"
    print(f"  [{status}] {required}")

In [ ]:
import torch

from conv2d_pitch_only import Conv2dPitchOnlyModel
from foundational_heads import find_cqt_jams_pairs, cross_validate_by_player_single_task

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
if device.type != "cuda":
    print("WARNING: not using CUDA -- if this is a RunPod GPU pod, training will be much slower than expected. Check the pod's PyTorch/CUDA install.")

## Hyperparameters
Same values currently in `conv2d_pitch_only.py`'s `main()` -- edit here to experiment without touching the .py file.

In [ ]:
input_bins = 252
sr = 22050
hop_length = 512
time_frames = round(4 * sr / hop_length)   # 4s sliding window
stride = round(1 * sr / hop_length)        # 1s stride
num_pitches = 49
batch_size = 16
temporal_hidden = 256
latent_dim = 128
dropout = 0.4

epochs = 100
lr = 0.001
patience = 10

In [ ]:
cqt_files, jams_files = find_cqt_jams_pairs()
print(f"{len(cqt_files)} debleeded recordings paired")

In [ ]:
def build_model():
    return Conv2dPitchOnlyModel(
        input_bins=input_bins,
        num_pitches=num_pitches,
        temporal_hidden=temporal_hidden,
        latent_dim=latent_dim,
        dropout=dropout,
    )

# quick sanity check before committing to the full run
_m = build_model().to(device)
_x = torch.randn(2, 1, time_frames, input_bins).to(device)
_out = _m(_x)
print("forward pass OK, output shape:", tuple(_out.shape))
print("params:", sum(p.numel() for p in _m.parameters()))
del _m, _x, _out

## Run the job
This trains 6 models (one per held-out player) -- expect this cell to run for a while. Progress prints per-epoch as it goes (streams live in the browser output, same as any notebook cell), so you'll see it incrementally rather than only at the end. Per-fold checkpoints are saved to `output/conv2d_pitch_only_checkpoint_holdout{player}.pt` as each fold finishes.

**Don't close the browser tab while this runs** -- if you do, the kernel keeps running on the pod regardless (it's server-side, not tied to your browser connection), but you won't see live output until you reconnect. Losing your local internet connection is fine for the same reason; losing the *pod* (stopped/terminated) is not.

In [ ]:
results = cross_validate_by_player_single_task(
    model_fn=build_model,
    cqt_files=cqt_files,
    jams_files=jams_files,
    task="pitch",
    time_frames=time_frames,
    stride=stride,
    batch_size=batch_size,
    epochs=epochs,
    lr=lr,
    patience=patience,
    device=device,
    checkpoint_dir="output",
    checkpoint_prefix="conv2d_pitch_only_checkpoint",
    extra_checkpoint_fields={
        "input_bins": input_bins,
        "num_pitches": num_pitches,
        "temporal_hidden": temporal_hidden,
        "latent_dim": latent_dim,
        "dropout": dropout,
    },
)
print("Cross-validation job successful.")

## Inspect results
`results` is a list of per-fold metric dicts (`held_out_player`, `loss`, `f1`, `precision`, `recall`) -- same values already printed in the summary above, kept here for further analysis/plotting if you want it.

In [ ]:
import pandas as pd
pd.DataFrame(results)